## Imports

In [1]:
from   pathlib import Path
import matplotlib.pyplot as plt
import pickle
import torch
import pandas as pd
# import mlflow

from neuralhydrology.evaluation     import metrics
from neuralhydrology.nh_run         import start_run, eval_run
from neuralhydrology.datasetzoo     import register_dataset
from neuralhydrology.utils.config   import Config

from custom.dataset.swisshourly import SwissHourly
register_dataset("swisshourly", SwissHourly)

# Custom functions plot_training_curves
from custom.functions.Plot import plot_training_curves, plot_validation_metrics
from custom.functions.Plot import plot_mape_horizon, plot_forecast_24h, plot_forecast_horizon_24h,plot_forecast_horizon_series
from custom.functions.utils_fla import parse_neuralhydrology_log, hydrograph_flows_distribution, print_hydrograph_distribution
from custom.functions.utils_fla import info_results, mape_array, dict_without_xr_key



In [2]:
runFolder = Path("./runs/mape8_base")
basin = "massa_with_RS"

model_str = f"model_from_best"
testPath = Path(runFolder / "test" / f"{model_str}" / "test_results.p")
with open(testPath, "rb") as fp:
    results = pickle.load(fp)

results_ = results[basin]["1h"]["xr"]
scores = dict_without_xr_key(results[basin]['1h'])

# store only the values in numpy arrays from xarray DataArrays
y_true = results_['streamflow_obs']
y_pred = results_['streamflow_sim']

In [3]:
# print(y_pred[1,-1])
# print(y_pred[2,-1])

In [4]:

# res = hydrograph_flows_distribution(y_true[:,0])


# print_hydrograph_distribution(res)


In [5]:
# Replace with your file path
file_path = r"..\..\data\massa_with_RS.csv"

# Load the CSV
df = pd.read_csv(file_path)

streamflow = df['streamflow']
simulation = df['Massa|22_Q simule']
res_meas = hydrograph_flows_distribution(streamflow)
res_sim = hydrograph_flows_distribution(simulation)

print("Measured Flow:")
print_hydrograph_distribution(res_meas)

print("\nSimulated Flow:")
print_hydrograph_distribution(res_sim)


Measured Flow:
Total points: 358753

Class           Count   Percentage
----------------------------------
0–10           221416       61.72%
10–40           85520       23.84%
40–80           48852       13.62%
80–∞             2965        0.83%

Simulated Flow:
Total points: 358753

Class           Count   Percentage
----------------------------------
0–10           213439       59.49%
10–40           95710       26.68%
40–80           47896       13.35%
80–∞             1708        0.48%


In [6]:
from custom.functions.utils_fla import mape_by_flow_class
import numpy as np

res = mape_by_flow_class(y_true[:, -1], y_pred[:,-1])

print(f"Global MAPE: {res['global_mape']:.2f}%\n")

# nice aligned printout
labels = res["labels"]
counts = res["counts"]
valid_counts = res["valid_counts"]
mape = res["mape_per_class"]

width_label = max(len("Class"), *(len(s) for s in labels))

header = (
    f"{'Class':<{width_label}} "
    f"{'Count':>10} "
    f"{'Valid':>10} "
    f"{'MAPE (%)':>12}"
)
print(header)
print("-" * len(header))

for lab, c, vc, m in zip(labels, counts, valid_counts, mape):
    m_str = f"{m:0.2f}" if np.isfinite(m) else "nan"
    print(f"{lab:<{width_label}} {c:>10d} {vc:>10d} {m_str:>12}")


Global MAPE: 8.88%

Class      Count      Valid     MAPE (%)
----------------------------------------
0–10        5007       5007         9.03
10–40       2067       2067         8.20
40–80       1501       1501         9.00
80–∞         210        210        11.24
